generate random walk baseline forecasts for all disease outcomes for horizons 1-12 weeks ahead

In [1]:
import pandas as pd
import numpy as np
import datetime
from epiweeks import Week
import datetime
import random
from datetime import timedelta
import time
import warnings
warnings.filterwarnings('ignore')

In [2]:
import matplotlib.dates as mdates
def set_date_axis_fmt(ax):
    # Set the locator
    locator = mdates.MonthLocator(interval=2)  # every month
    # Specify the format
    fmt = mdates.DateFormatter('%b %y')

    X = ax.xaxis
    X.set_major_locator(locator)
    # Specify formatter
    X.set_major_formatter(fmt)
 

# Flu hospitalizations

In [4]:
# flu hospitalizations baseline model        
        
def log_flu_hosp_baseline(observations, dates, locations, use_submitted_median=True):

    locationsdf = pd.read_csv('dat/locations.csv')
    locationsdf['location'] = locationsdf['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 
    
    baseline_model_all = pd.DataFrame()
    for date in dates:
        print(date)
        for location in locations:
            
            if location in ['GU', 'MP', 'AS']:
                continue

            forecast_date = pd.to_datetime(date)
            week_end = Week.fromdate(forecast_date).enddate() 

            last_monday_forecast = pd.to_datetime("2023-05-22")

            # filter surveillance data for chosen location and for all time points prior to forecast date

            if forecast_date <= last_monday_forecast:
                obs_filt = observations[(observations['location'] == location) &\
                            (observations['target_end_date'] <= forecast_date - timedelta(days=2)) &\
                            (observations['as_of'] == forecast_date)].sort_values(by='week_end',ascending=True)
            elif forecast_date > pd.to_datetime('2024-10-01'):
                obs_filt = observations[(observations['location'] == location) &\
                            (observations['target_end_date'] < forecast_date - timedelta(days=3)) &\
                            (observations['as_of'] == forecast_date - timedelta(days=3))&\
                            (observations.week_end>=pd.to_datetime('2022-09-12'))].sort_values(by='week_end',ascending=True)
            else:
                obs_filt = observations[(observations['location'] == location) &\
                            (observations['target_end_date'] < forecast_date - timedelta(days=3)) &\
                            (observations['as_of'] == forecast_date + timedelta(days=4))].sort_values(by='week_end',ascending=True)


            if len(obs_filt)==0:
                print('no surveillance data')


            # get list of differences between successive weeks from historical incidence data
                # window_size = how much historical data to use. None = use all historical data. 
                # symmetrize: (logical) True = look at differences and their negatives to get symmetric distribution

            window_size = None
            symmetrize=True

            if window_size is None:
                window_size = len(obs_filt) - 1
            if window_size >= len(obs_filt):
                window_size = len(obs_filt) - 1

            diffs = np.diff(obs_filt['value'])[-window_size:]
            diffs = diffs[~np.isnan(diffs)]

            if symmetrize:
                inc_diffs = list(diffs) + [-x for x in diffs]
            else:
                inc_diffs = list(diffs)

            inc_diffs = np.array(inc_diffs)


            # compute quantiles of differences distribution and ensure the median difference is 0. 

            num_samples = 100000

            sampled_inc_diffs = np.quantile(inc_diffs, np.linspace(0, 1, num_samples))

            median_diff = np.median(sampled_inc_diffs)
            sampled_inc_diffs -= median_diff


            # add differences to last observed surveillance data point and ensure non-negative incidence values

            inc_data = obs_filt['value']
            
            # ensure median is same as submitted Flusight baseline
            if use_submitted_median == True:
                if forecast_date < last_monday_forecast:
                    df = pd.read_csv(f'https://raw.githubusercontent.com/cdcepi/Flusight-forecast-data/refs/heads/master/data-forecasts/Flusight-baseline/{date}-Flusight-baseline.csv')
                    df = df.merge(locationsdf, on='location')
                    last_observed_value = df[(df.abbreviation==location) & (df['quantile']==0.5)].value.unique()[0]
                    last_observed_value = np.log(last_observed_value+1)
                else:
                    if location=='VI':
                        last_observed_value = inc_data.iloc[-1]
                    else:
                        df = pd.read_csv(f'https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/refs/heads/main/model-output/FluSight-baseline/{date}-FluSight-baseline.csv')
                        df = df.merge(locationsdf, on='location')
                        df['output_type_id'] = df['output_type_id'].astype(str)
                        last_observed_value = df[(df.abbreviation==location) & (df.output_type_id=='0.5')].value.unique()[0]
                        last_observed_value = np.log(last_observed_value+1)
            else:
                last_observed_value = inc_data.iloc[-1]
                
            sampled_inc_raw = last_observed_value + sampled_inc_diffs
            sampled_inc_raw = np.maximum(sampled_inc_raw, 0)
            


            # calculate quantiles from sampled incidence values and save first horizon

            quantiles = [0.01, 0.025] + list(np.arange(0.05, 0.96, 0.05)) + [0.975, 0.99]

            baseline_model = pd.DataFrame()

            predictions = np.quantile(sampled_inc_raw, quantiles, axis=0)
            

            results = pd.DataFrame({'horizon':1, 'forecast_date':pd.to_datetime(forecast_date),
                                    'target_end_date': week_end- pd.Timedelta(days=7) + timedelta(weeks=1)  ,
                                    'target':'inc hosp','location':location ,'quantile':quantiles, 'value':predictions})

            baseline_model = pd.concat([baseline_model, results])


            #  get predictions for larger horizons

            horizon = 12 # number of weeks ahead to predict

            for h in list(range(2, horizon+1)):
                # Sample from differences again with replacement for each new horizon, adds uncertainty at each horizon
                sampled_inc_diffs = np.random.choice(sampled_inc_diffs, size=num_samples, replace=True)
                sampled_inc_raw = sampled_inc_raw + sampled_inc_diffs # add to previous horizon incidence counts

                # Adjust incidence if symmetrize is TRUE. ensures median is same as last observed surveillance
                if symmetrize:
                    sampled_inc_corrected = sampled_inc_raw - (np.median(sampled_inc_raw) - last_observed_value)
                else:
                    sampled_inc_corrected = sampled_inc_raw

                # Ensure non-negative incidence
                sampled_inc_corrected = np.maximum(sampled_inc_corrected, 0)

                predictionshorizon = np.quantile(sampled_inc_corrected, quantiles, axis=0) # compute quantiles

                # save incidence for new horizon
                resultshorizon = pd.DataFrame({'horizon':h, 'forecast_date':pd.to_datetime(forecast_date),
                                                'target_end_date':week_end- pd.Timedelta(days=7) + timedelta(weeks=1*h),
                                                'target':'inc hosp','location':location , 'quantile':quantiles, 
                                                'value':predictionshorizon})

                baseline_model = pd.concat([baseline_model, resultshorizon])

            baseline_model['quantile'] = baseline_model['quantile'].round(3)  

            baseline_model_all = pd.concat([baseline_model_all, baseline_model])
            
    baseline_model_all['value'] = baseline_model_all['value'].apply(lambda x: np.exp(x))       
            
    return baseline_model_all



In [5]:
#  get historical data

obs_versioned1 = (pd.read_parquet('surveillance data/versioned_data/influenza_hosp.parquet')
    .assign(value=lambda df: df['value'].where(df['value'] >= 0, pd.NA))
    .assign(target_end_date=lambda df: pd.to_datetime(df['time_value']).dt.to_period('W-SAT').dt.end_time)
    .assign(target_end_date=lambda df: df['target_end_date'] - timedelta(days=1))
    .groupby(['as_of', 'geo_value', 'target_end_date'], as_index=False)
    .agg(value=('value', lambda x: x.sum(skipna=True)))
    .sort_values('geo_value'))
obs_versioned1 = obs_versioned1[obs_versioned1.target_end_date > pd.to_datetime('2022-01-01')]

obs_versioned2 = (pd.read_parquet("surveillance data/versioned_data/influenza_hosp_24-26.parquet").assign(value=lambda df: \
                                                            df['value'].where(df['value'] >= 0, pd.NA))
    .assign(target_end_date=lambda df: df['time_value'] + timedelta(days=6))
    .sort_values('geo_value'))

obs_versioned2 = obs_versioned2[obs_versioned2.target_end_date > pd.to_datetime('2022-02-12')]



obs_versioned = pd.concat([obs_versioned1, obs_versioned2])
obs_versioned['year'] = obs_versioned['target_end_date'].apply(lambda x: x.year)
obs_versioned['epiweek'] = obs_versioned['target_end_date'].apply(lambda x: Week.fromdate(x))
obs_versioned['week'] = obs_versioned['epiweek'].apply(lambda x: str(x)[4:])


obs_versioned['epiweek_start_date'] = obs_versioned['target_end_date'].apply(lambda row: Week.fromdate(row).startdate())
obs_versioned['epiweek_end_date'] = obs_versioned['target_end_date'].apply(lambda row: Week.fromdate(row).enddate())

obs_versioned['as_of_year'] = obs_versioned['as_of'].apply(lambda x: x.year)
obs_versioned['as_of_week'] = obs_versioned['as_of'].apply(lambda x: str(Week.fromdate(x))[4:])
obs_versioned['as_of_epiweek_end_date'] = obs_versioned.apply(lambda row: Week(int(row['as_of_year']), 
                                                                                 int(row['as_of_week'])).enddate(), axis=1)
obs_versioned = obs_versioned.drop(columns=['year', 'week', 'as_of_week', 'as_of_year'])
obs_versioned = obs_versioned.rename(columns={'epiweek_start_date': 'week_start', 'epiweek_end_date': 'week_end'})
obs_versioned['geo_value'] = obs_versioned['geo_value'].apply(lambda x: str.upper(x))

obs_versioned = obs_versioned.rename(columns={'geo_value':'location'})

observations = obs_versioned.rename(columns={'geo_value':'location'}).drop(columns=['signal', 'source', 
                                                                                    'time_type','issue'])

observations['value'] = observations['value'].apply(lambda x: np.log(x+1))


dates = [ '2022-01-17', '2022-01-24', '2022-01-31', '2022-02-07', '2022-02-14', '2022-02-21',
         '2022-02-28', '2022-03-07', '2022-03-14', '2022-03-21', '2022-03-28', '2022-04-04', '2022-04-11',
         '2022-04-18', '2022-04-25', '2022-05-02', '2022-05-09', '2022-05-16', '2022-05-23', '2022-05-30',
         '2022-06-06', '2022-06-13', '2022-06-20', '2022-10-03', '2022-10-17', '2022-10-24', '2022-10-31',
         '2022-11-07', '2022-11-14', '2022-11-21', '2022-11-28', '2022-12-05', '2022-12-12', '2022-12-19',
         '2022-12-26', '2023-01-02', '2023-01-09', '2023-01-16', '2023-01-23', '2023-01-30', '2023-02-06',
         '2023-02-13', '2023-02-20', '2023-02-27', '2023-03-06', '2023-03-13', '2023-03-20', '2023-03-27',
         '2023-04-03', '2023-04-10', '2023-04-17', '2023-04-24', '2023-05-01', '2023-05-08', '2023-05-15']
        
            
locations = observations.location.unique()


In [ ]:
baseline_model_fluhosp_1 = log_flu_hosp_baseline(observations, dates, locations)

2022-01-17
2022-01-24
2022-01-31
2022-02-07
2022-02-14
2022-02-21
2022-02-28
2022-03-07
2022-03-14
2022-03-21
2022-03-28
2022-04-04
2022-04-11
2022-04-18
2022-04-25
2022-05-02
2022-05-09
2022-05-16
2022-05-23
2022-05-30
2022-06-06
2022-06-13
2022-06-20
2022-10-03
2022-10-17
2022-10-24
2022-10-31
2022-11-07
2022-11-14
2022-11-21
2022-11-28
2022-12-05
2022-12-12
2022-12-19
2022-12-26
2023-01-02
2023-01-09
2023-01-16
2023-01-23
2023-01-30
2023-02-06
2023-02-13
2023-02-20


In [36]:
# get historical data, current and past  

obs_versioned1 = (pd.read_parquet('surveillance data/versioned_data/influenza_hosp.parquet')
    .assign(value=lambda df: df['value'].where(df['value'] >= 0, pd.NA))
    .assign(target_end_date=lambda df: pd.to_datetime(df['time_value']).dt.to_period('W-SAT').dt.end_time)
    .assign(target_end_date=lambda df: df['target_end_date'] - timedelta(days=1))
    .groupby(['as_of', 'geo_value', 'target_end_date'], as_index=False)
    .agg(value=('value', lambda x: x.sum(skipna=True)))
    .sort_values('geo_value'))
obs_versioned1 = obs_versioned1[obs_versioned1.target_end_date > pd.to_datetime('2022-01-01')]

obs_versioned2 = (pd.read_parquet("surveillance data/versioned_data/influenza_hosp_24-26.parquet").assign(value=lambda df: \
                                                            df['value'].where(df['value'] >= 0, pd.NA))
    .assign(target_end_date=lambda df: df['time_value'] + timedelta(days=6))
    .sort_values('geo_value'))

obs_versioned2 = obs_versioned2[obs_versioned2.target_end_date > pd.to_datetime('2022-02-12')]


observations2a = obs_versioned2[(obs_versioned2.target_end_date<=pd.to_datetime('2024-05-04')) ]
observations2b = obs_versioned2[(obs_versioned2.target_end_date>=pd.to_datetime('2024-11-11'))]
observations2 = pd.concat([observations2a, observations2b])


obs_versioned = pd.concat([obs_versioned1, observations2])
obs_versioned['year'] = obs_versioned['target_end_date'].apply(lambda x: x.year)
obs_versioned['epiweek'] = obs_versioned['target_end_date'].apply(lambda x: Week.fromdate(x))
obs_versioned['week'] = obs_versioned['epiweek'].apply(lambda x: str(x)[4:])


obs_versioned['epiweek_start_date'] = obs_versioned['target_end_date'].apply(lambda row: Week.fromdate(row).startdate())
obs_versioned['epiweek_end_date'] = obs_versioned['target_end_date'].apply(lambda row: Week.fromdate(row).enddate())

obs_versioned['as_of_year'] = obs_versioned['as_of'].apply(lambda x: x.year)
obs_versioned['as_of_week'] = obs_versioned['as_of'].apply(lambda x: str(Week.fromdate(x))[4:])
obs_versioned['as_of_epiweek_end_date'] = obs_versioned.apply(lambda row: Week(int(row['as_of_year']), 
                                                                                 int(row['as_of_week'])).enddate(), axis=1)
obs_versioned = obs_versioned.drop(columns=['year', 'week', 'as_of_week', 'as_of_year'])
obs_versioned = obs_versioned.rename(columns={'epiweek_start_date': 'week_start', 'epiweek_end_date': 'week_end'})
obs_versioned['geo_value'] = obs_versioned['geo_value'].apply(lambda x: str.upper(x))

obs_versioned = obs_versioned.rename(columns={'geo_value':'location'})

observations = obs_versioned.rename(columns={'geo_value':'location'}).drop(columns=['signal', 'source', 
                                                                                    'time_type','issue'])



observations['value'] = observations['value'].apply(lambda x: np.log(x+1))

dates = ['2023-10-14', '2023-10-21', '2023-10-28', '2023-11-04', '2023-11-11',  '2023-11-18',
         '2023-11-25', '2023-12-02', '2023-12-09','2023-12-16', '2023-12-23', '2023-12-30', '2024-01-06',
         '2024-01-13', '2024-01-20', '2024-01-27', '2024-02-03', '2024-02-10', '2024-02-17', '2024-02-24',
         '2024-03-02', '2024-03-09', '2024-03-16', '2024-03-23', '2024-03-30', '2024-04-06', '2024-04-13',
         '2024-04-20', '2024-04-27', '2024-05-04', '2024-11-23', '2024-11-30', '2024-12-07', '2024-12-14', 
         '2024-12-21', '2024-12-28', '2025-01-04', '2025-01-11', '2025-01-18', '2025-02-01', '2025-02-08', 
         '2025-02-15', '2025-02-22', '2025-03-01', '2025-03-08', '2025-03-15', '2025-03-22', '2025-03-29', 
         '2025-04-05', '2025-04-12', '2025-04-19', '2025-04-26', '2025-05-03', '2025-05-10', '2025-05-17', 
         '2025-05-24', '2025-05-31', '2025-11-22', '2025-11-29', '2025-12-06', '2025-12-13', '2025-12-20', 
         '2025-12-27', '2026-01-03', '2026-01-10', '2026-01-17', '2026-01-24', '2026-01-31', '2026-02-07', 
         '2026-02-14', '2026-02-21', '2026-02-28', '2026-03-07', '2026-03-14', '2026-03-21', '2026-03-28', 
         '2026-04-04', '2026-04-11', '2026-04-18', '2026-04-25', '2026-05-02', '2026-05-09', '2026-05-16', 
         '2026-05-23', '2026-05-30']  

locations = observations.location.unique()

baseline_model_fluhosp2 = log_flu_hosp_baseline(observations, dates, locations)


2023-10-14
2023-10-21
2023-10-28
2023-11-04
2023-11-11
2023-11-18
2023-11-25
2023-12-02
2023-12-09
2023-12-16
2023-12-23
2023-12-30
2024-01-06
2024-01-13
2024-01-20
2024-01-27
2024-02-03
2024-02-10
2024-02-17
2024-02-24
2024-03-02
2024-03-09
2024-03-16
2024-03-23
2024-03-30
2024-04-06
2024-04-13
2024-04-20
2024-04-27
2024-05-04
2024-11-23
2024-11-30
2024-12-07
2024-12-14
2024-12-21
2024-12-28
2025-01-04
2025-01-11
2025-01-18
2025-02-01
2025-02-08
2025-02-15
2025-02-22
2025-03-01
2025-03-08
2025-03-15
2025-03-22
2025-03-29
2025-04-05
2025-04-12
2025-04-19
2025-04-26
2025-05-03
2025-05-10
2025-05-17
2025-05-24
2025-05-31


In [38]:
baseline_model_fluhosp = pd.concat([baseline_model_fluhosp_1, baseline_model_fluhosp2])
baseline_model_fluhosp.to_parquet('benchmark forecasts/random walk baseline/flu_hosp_log_baseline_h12.pq')

In [ ]:
fluhosp = pd.read_parquet('benchmark forecasts/random walk baseline/flu_hosp_log_baseline_h12.pq').rename(columns={'location':'abbreviation'})

locationsdf = pd.read_csv('dat/locations.csv')
locationsdf['location'] = locationsdf['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 
locationsdf = locationsdf[['abbreviation', 'location', 'location_name']]

fluhosp = fluhosp.merge(locationsdf, on=['abbreviation'])

fluhosp['reference_date'] = fluhosp['forecast_date'].apply(lambda x: x if x>pd.to_datetime('2023-09-01') else np.nan)
fluhosp['forecast_date'] = fluhosp['forecast_date'].apply(lambda x: x if x<pd.to_datetime('2023-09-01') else np.nan)

fluhosp.to_parquet('benchmark forecasts/random walk baseline/flu_hosp_log_baseline_h12.pq')

# COVID cases

In [177]:
# get historical data, current and past 
use_submitted_median = True

locationsdf = pd.read_csv('dat/locations.csv')
locationsdf['location'] = locationsdf['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 
    

df_case_versioned = pd.read_parquet('surveillance data/versioned_data/covid_cases.parquet')

df_case_versioned_agg = (df_case_versioned.assign(value=lambda x: x['value'].where(x['value'] >= 0))
    .assign(target_end_date=lambda x: pd.to_datetime(x['time_value']).dt.to_period('W-SAT').dt.to_timestamp('W-SAT'))
    .groupby(['as_of', 'geo_value', 'target_end_date'], dropna=False).agg({'value': 'sum'}).reset_index()
    .sort_values('geo_value').reset_index(drop=True))

observations = df_case_versioned_agg[df_case_versioned_agg.target_end_date>=pd.to_datetime('2020-01-01')].rename(columns={'geo_value':'location'})
observations['week_end'] = observations['target_end_date'].apply(lambda x: Week.fromdate(x).enddate())
observations['location'] = observations['location'].apply(lambda x: str.upper(x))

observations['value'] = observations['value'].apply(lambda x: np.log(x+1))

observations_idx = (observations.sort_values(['location', 'as_of', 'target_end_date']).set_index(['location', 
                                                                                                  'as_of']))

      
df_case = (pd.read_parquet('surveillance data/observed data/truth_cases.parquet')
           .assign(obs_value=lambda x: x['value'].where(x['value'] >= 0)).sort_values('target_end_date')
    .query("geo_type == 'state'").reset_index(drop=True))

df_case = df_case[(df_case.target_end_date>pd.to_datetime('2020-04-06')) &\
                  (df_case.target_end_date<=pd.to_datetime('2021-12-31'))]

forecast_dates_cases = df_case['target_end_date'].unique() - pd.Timedelta(days=5)


# choose forecast date and location. 

baseline_model_all = pd.DataFrame()
for date in forecast_dates_cases:
    print(date)
    
    dfsubmitted = pd.read_csv(f'https://raw.githubusercontent.com/reichlab/covid19-forecast-hub/refs/heads/master/data-processed/COVIDhub-baseline/{date}-COVIDhub-baseline.csv')
    dfsubmitted = dfsubmitted.merge(locationsdf, on='location')
    dfsubmitted = dfsubmitted[dfsubmitted.target.isin(["1 wk ahead inc case", "2 wk ahead inc case", 
                                                       "3 wk ahead inc case",  "4 wk ahead inc case"])]
    dfsubmitted['quantile'] = dfsubmitted['quantile'].astype(str)
    
    for location in observations.location.unique()  :
        
        if location in ['GU', 'MP',  'AS']:
            continue

        forecast_date = pd.to_datetime(date)
        week_end = Week.fromdate(forecast_date).enddate() 

        last_monday_forecast = pd.to_datetime("2023-05-22")

            
        if forecast_date==pd.to_datetime('2020-04-06') :
            continue

        # filter surveillance data for chosen location and for all time points prior to forecast date

            
        try:
            obs_filt = observations_idx.loc[(location, forecast_date)]
        except KeyError:
            continue

        obs_filt = (obs_filt.loc[obs_filt['target_end_date'] <= forecast_date - timedelta(days=2)]
            .sort_values('week_end').dropna(subset=['value']).reset_index(drop=True))    

            
        if len(obs_filt)==0:
            print('no surveillance data')


        # get list of differences between successive weeks from historical incidence data
            # window_size = how much historical data to use. None = use all historical data. 
            # symmetrize: (logical) True = look at differences and their negatives to get symmetric distribution
            
        window_size = None
        symmetrize=True

        if window_size is None:
            window_size = len(obs_filt) - 1
        if window_size >= len(obs_filt):
            window_size = len(obs_filt) - 1

        diffs = np.diff(obs_filt['value'])[-window_size:]
        diffs = diffs[~np.isnan(diffs)]

        if symmetrize:
            inc_diffs = list(diffs) + [-x for x in diffs]
        else:
            inc_diffs = list(diffs)

        inc_diffs = np.array(inc_diffs)


        #  compute quantiles of differences distribution and ensure the median difference is 0. 

        num_samples = 100000

        sampled_inc_diffs = np.quantile(inc_diffs, np.linspace(0, 1, num_samples))

        median_diff = np.median(sampled_inc_diffs)
        sampled_inc_diffs -= median_diff


        # add differences to last observed surveillance data point and ensure non-negative incidence values

        inc_data = obs_filt['value']
        
        if use_submitted_median == True:
            last_observed_value = dfsubmitted[(dfsubmitted.abbreviation==location) &\
                                              (dfsubmitted['quantile']=='0.5')].value.unique()[0]
            last_observed_value = np.log(last_observed_value+1)
        else:
            last_observed_value = inc_data.iloc[-1]


        sampled_inc_raw = last_observed_value + sampled_inc_diffs
        sampled_inc_raw = np.maximum(sampled_inc_raw, 0)


        # calculate quantiles from sampled incidence values and save first horizon

        quantiles = [0.025, 0.1  , 0.25 , 0.5  , 0.75 , 0.9  , 0.975]

        baseline_model = pd.DataFrame()

        predictions = np.quantile(sampled_inc_raw, quantiles, axis=0)

        results = pd.DataFrame({'horizon':1, 'forecast_date':pd.to_datetime(forecast_date),
                                'target_end_date': forecast_date + pd.Timedelta(days=5)   ,
                                'target':'inc case','location':location ,'quantile':quantiles, 'value':predictions})

        baseline_model = pd.concat([baseline_model, results])


        # get predictions for larger horizons

        horizon = 12 # number of weeks ahead to predict

        for h in list(range(2, horizon+1)):
            # Sample from differences again with replacement for each new horizon 
            sampled_inc_diffs = np.random.choice(sampled_inc_diffs, size=num_samples, replace=True)
            sampled_inc_raw = sampled_inc_raw + sampled_inc_diffs # add to previous horizon incidence counts

            # if symmetrize is TRUE, ensures median is same as last observed surveillance
            if symmetrize:
                sampled_inc_corrected = sampled_inc_raw - (np.median(sampled_inc_raw) - last_observed_value)
            else:
                sampled_inc_corrected = sampled_inc_raw

            # Ensure non-negative incidence
            sampled_inc_corrected = np.maximum(sampled_inc_corrected, 0)

            predictionshorizon = np.quantile(sampled_inc_corrected, quantiles, axis=0) # compute quantiles

            # save incidence for new horizon
            resultshorizon = pd.DataFrame({'horizon':h, 'forecast_date':pd.to_datetime(forecast_date),
                                            'target_end_date':forecast_date+ pd.Timedelta(days=5) + timedelta(weeks=1*(h-1)),
                                            'target':'inc case','location':location , 'quantile':quantiles, 
                                            'value':predictionshorizon})

            baseline_model = pd.concat([baseline_model, resultshorizon])

        baseline_model['quantile'] = baseline_model['quantile'].round(3)  

        baseline_model_all = pd.concat([baseline_model_all, baseline_model])
        
        
baseline_model_all['value'] = baseline_model_all['value'].apply(lambda x: np.exp(x))       
    
baseline_model_all.to_parquet('benchmark forecasts/random walk baseline/covid_case_log_baseline_h12.pq')

2020-04-06
2020-04-13
2020-04-20
2020-04-27
2020-05-04
2020-05-11
2020-05-18
2020-05-25
2020-06-01
2020-06-08
2020-06-15
2020-06-22
2020-06-29
2020-07-06
2020-07-13
2020-07-20
2020-07-27
2020-08-03
2020-08-10
2020-08-17
2020-08-24
2020-08-31
2020-09-07
2020-09-14
2020-09-21
2020-09-28
2020-10-05
2020-10-12
2020-10-19
2020-10-26
2020-11-02
2020-11-09
2020-11-16
2020-11-23
2020-11-30
2020-12-07
2020-12-14
2020-12-21
2020-12-28
2021-01-04
2021-01-11
2021-01-18
2021-01-25
2021-02-01
2021-02-08
2021-02-15
2021-02-22
2021-03-01
2021-03-08
2021-03-15
2021-03-22
2021-03-29
2021-04-05
2021-04-12
2021-04-19
2021-04-26
2021-05-03
2021-05-10
2021-05-17
2021-05-24
2021-05-31
2021-06-07
2021-06-14
2021-06-21
2021-06-28
2021-07-05
2021-07-12
2021-07-19
2021-07-26
2021-08-02
2021-08-09
2021-08-16
2021-08-23
2021-08-30
2021-09-06
2021-09-13
2021-09-20
2021-09-27
2021-10-04
2021-10-11
2021-10-18
2021-10-25
2021-11-01
2021-11-08
2021-11-15
2021-11-22
2021-11-29
2021-12-06
2021-12-13
2021-12-20


In [ ]:
covcase = pd.read_parquet('benchmark forecasts/random walk baseline/covid_case_log_baseline_h12.pq')

locationsdf = pd.read_csv('dat/locations.csv')
locationsdf['location'] = locationsdf['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 
locationsdf = locationsdf[['abbreviation', 'location', 'location_name']]

covcase = covcase.merge(locationsdf, on=['abbreviation', 'location', 'location_name'])

covcase.to_parquet('benchmark forecasts/random walk baseline/covid_case_log_baseline_h12.pq')

# COVID deaths

In [178]:
#  get historical data, current and past

use_submitted_median = True

locationsdf = pd.read_csv('dat/locations.csv')
locationsdf['location'] = locationsdf['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 


df_death_versioned = pd.read_parquet('surveillance data/versioned_data/covid_deaths.parquet')

df_death_versioned_agg = (df_death_versioned.assign(value=lambda x: x['value'].where(x['value'] >= 0))
    .assign(target_end_date=lambda x: pd.to_datetime(x['time_value']).dt.to_period('W-SAT').dt.to_timestamp('W-SAT'))
    .groupby(['as_of', 'geo_value', 'target_end_date'], dropna=False).agg({'value': 'sum'}).reset_index()
    .sort_values('geo_value').reset_index(drop=True))

observations = df_death_versioned_agg[df_death_versioned_agg.target_end_date>=pd.to_datetime('2020-01-01')].rename(columns={'geo_value':'location'})
observations['week_end'] = observations['target_end_date'].apply(lambda x: Week.fromdate(x).enddate())
observations['location'] = observations['location'].apply(lambda x: str.upper(x))

observations['value'] = observations['value'].apply(lambda x: np.log(x+1))


observations_idx = (observations.sort_values(['location', 'as_of', 'target_end_date']).set_index(['location', 
                                                                                                  'as_of']))

      
df_death = (pd.read_parquet('surveillance data/observed data/truth_deaths.parquet')
           .assign(obs_value=lambda x: x['value'].where(x['value'] >= 0)).sort_values('target_end_date')
    .query("geo_type == 'state'").reset_index(drop=True))

df_death = df_death[(df_death.target_end_date>=pd.to_datetime('2020-04-06')) &\
                  (df_death.target_end_date<=pd.to_datetime('2022-12-31'))]

forecast_dates_death = df_death['target_end_date'].unique() - pd.Timedelta(days=5)


#  choose forecast date and location. 

baseline_model_all = pd.DataFrame()
for date in forecast_dates_death:
    print(date)
    
    dfsubmitted = pd.read_csv(f'https://raw.githubusercontent.com/reichlab/covid19-forecast-hub/refs/heads/master/data-processed/COVIDhub-baseline/{date}-COVIDhub-baseline.csv')
    dfsubmitted = dfsubmitted.merge(locationsdf, on='location')
    dfsubmitted = dfsubmitted[dfsubmitted.target.isin(["1 wk ahead inc death", "2 wk ahead inc death", 
                                                       "3 wk ahead inc death",  "4 wk ahead inc death"])]
    dfsubmitted['quantile'] = dfsubmitted['quantile'].astype(str)

    for location in observations.location.unique()  :
        
        if location in ['GU', 'MP', 'AS']:
            continue

        forecast_date = pd.to_datetime(date)
        week_end = Week.fromdate(forecast_date).enddate() 

    
        #location = 'US'
        if forecast_date==pd.to_datetime('2020-04-06') :
            continue

        # filter surveillance data for chosen location and for all time points prior to forecast date

          
        try:
            obs_filt = observations_idx.loc[(location, forecast_date)]
        except KeyError:
            continue

        obs_filt = (obs_filt.loc[obs_filt['target_end_date'] <= forecast_date - timedelta(days=2)]
            .sort_values('week_end').dropna(subset=['value']).reset_index(drop=True))    

        if len(obs_filt)==0:
            print('no surveillance data')


        #  get list of differences between successive weeks from historical incidence data
            # window_size = how much historical data to use. None = use all historical data. 
            # symmetrize: (logical) True = look at differences and their negatives to get symmetric distribution
          
        window_size = None
        symmetrize=True

        if window_size is None:
            window_size = len(obs_filt) - 1
        if window_size >= len(obs_filt):
            window_size = len(obs_filt) - 1

        diffs = np.diff(obs_filt['value'])[-window_size:]
        diffs = diffs[~np.isnan(diffs)]

        if symmetrize:
            inc_diffs = np.concatenate([diffs, -diffs])
        else:
            inc_diffs = list(diffs)

        inc_diffs = np.array(inc_diffs)


        # compute quantiles of differences distribution and ensure the median difference is 0. 

        num_samples = 100000

        sampled_inc_diffs = np.quantile(inc_diffs, np.linspace(0, 1, num_samples))

        median_diff = np.median(sampled_inc_diffs)
        sampled_inc_diffs -= median_diff


        # add differences to last observed surveillance data point and ensure non-negative incidence values

        inc_data = obs_filt['value'] 
        
        if use_submitted_median == True:
            last_observed_value = dfsubmitted[(dfsubmitted.abbreviation==location) & \
                                              (dfsubmitted['quantile']=='0.5')].value.unique()[0]
            last_observed_value = np.log(last_observed_value+1)
        else:
            last_observed_value = inc_data.iloc[-1]

        sampled_inc_raw = last_observed_value + sampled_inc_diffs
        sampled_inc_raw = np.maximum(sampled_inc_raw, 0)


        # calculate quantiles from sampled incidence values and save first horizon

        quantiles = [0.01, 0.025] + list(np.arange(0.05, 0.96, 0.05)) + [0.975, 0.99]

        predictions = np.quantile(sampled_inc_raw, quantiles, axis=0)

        results = pd.DataFrame({'horizon':1, 'forecast_date':pd.to_datetime(forecast_date),
                                'target_end_date': forecast_date + pd.Timedelta(days=5)   ,
                                'target':'inc death','location':location ,'quantile':quantiles, 'value':predictions})


        # get predictions for larger horizons

        horizon = 12 # number of weeks ahead to predict

        rows = []
        rows.append(results)
        
        for h in list(range(2, horizon+1)):
            # Sample from differences again with replacement for each new horizon 
            sampled_inc_diffs = np.random.choice(sampled_inc_diffs, size=num_samples, replace=True)
            sampled_inc_raw += sampled_inc_diffs # add to previous horizon incidence counts

            # Adjust incidence if symmetrize is TRUE. ensures median is same as last observed surveillance
            if symmetrize:
                sampled_inc_corrected = sampled_inc_raw - (np.median(sampled_inc_raw) - last_observed_value)
            else:
                sampled_inc_corrected = sampled_inc_raw

            # Ensure non-negative incidence
            sampled_inc_corrected = np.maximum(sampled_inc_corrected, 0)

            predictionshorizon = np.quantile(sampled_inc_corrected, quantiles, axis=0) # compute quantiles

            # save incidence for new horizon
            resultshorizon = pd.DataFrame({'horizon':h, 'forecast_date':pd.to_datetime(forecast_date),
                                            'target_end_date':forecast_date+ pd.Timedelta(days=5) + timedelta(weeks=1*(h-1)),
                                            'target':'inc death','location':location , 'quantile':quantiles, 
                                            'value':predictionshorizon})
            rows.append(resultshorizon)

        baseline_model = pd.concat(rows, ignore_index=True)

        baseline_model_all = pd.concat([baseline_model_all, baseline_model], ignore_index=True)

baseline_model_all['quantile'] = baseline_model_all['quantile'].round(3) 
        
baseline_model_all['value'] = baseline_model_all['value'].apply(lambda x: np.exp(x))       
        
baseline_model_all.to_parquet('benchmark forecasts/random walk baseline/covid_death_log_baseline_h12.pq')

2020-04-06
2020-04-13
2020-04-20
2020-04-27
2020-05-04
2020-05-11
2020-05-18
2020-05-25
2020-06-01
2020-06-08
2020-06-15
2020-06-22
2020-06-29
2020-07-06
2020-07-13
2020-07-20
2020-07-27
2020-08-03
2020-08-10
2020-08-17
2020-08-24
2020-08-31
2020-09-07
2020-09-14
2020-09-21
2020-09-28
2020-10-05
2020-10-12
2020-10-19
2020-10-26
2020-11-02
2020-11-09
2020-11-16
2020-11-23
2020-11-30
2020-12-07
2020-12-14
2020-12-21
2020-12-28
2021-01-04
2021-01-11
2021-01-18
2021-01-25
2021-02-01
2021-02-08
2021-02-15
2021-02-22
2021-03-01
2021-03-08
2021-03-15
2021-03-22
2021-03-29
2021-04-05
2021-04-12
2021-04-19
2021-04-26
2021-05-03
2021-05-10
2021-05-17
2021-05-24
2021-05-31
2021-06-07
2021-06-14
2021-06-21
2021-06-28
2021-07-05
2021-07-12
2021-07-19
2021-07-26
2021-08-02
2021-08-09
2021-08-16
2021-08-23
2021-08-30
2021-09-06
2021-09-13
2021-09-20
2021-09-27
2021-10-04
2021-10-11
2021-10-18
2021-10-25
2021-11-01
2021-11-08
2021-11-15
2021-11-22
2021-11-29
2021-12-06
2021-12-13
2021-12-20
2021-12-27

In [ ]:
covdeath = pd.read_parquet('benchmark forecasts/random walk baseline/covid_death_log_baseline_h12.pq').rename(columns={'location':'abbreviation'})

locationsdf = pd.read_csv('dat/locations.csv')
locationsdf['location'] = locationsdf['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 
locationsdf = locationsdf[['abbreviation', 'location', 'location_name']]

covdeath = covdeath.merge(locationsdf, on=['abbreviation'])

covdeath.to_parquet('benchmark forecasts/random walk baseline/covid_death_log_baseline_h12.pq')

# COVID daily hosp

In [58]:
# get historical data, current and past 

use_submitted_median = True

locationsdf = pd.read_csv('dat/locations.csv')
locationsdf['location'] = locationsdf['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 


df_hosp_versioned = pd.read_parquet('surveillance data/versioned_data/covid_hosp.parquet')

df_hosp_versioned_agg = (df_hosp_versioned.assign(value=lambda x: x['value'].where(x['value'] >= 0))
    .groupby(['as_of', 'geo_value', 'time_value'], dropna=False).agg({'value': 'sum'}).reset_index()
    .sort_values('geo_value').reset_index(drop=True))

observations = df_hosp_versioned_agg[df_hosp_versioned_agg.time_value>=pd.to_datetime('2020-01-01')].rename(columns={'geo_value':'location'})
observations['location'] = observations['location'].apply(lambda x: str.upper(x))
observations = observations[observations.time_value>= pd.to_datetime('2020-11-01')]

observations['value'] = observations['value'].apply(lambda x: np.log(x+1))

observations_idx = (observations.sort_values(['location', 'as_of', 'time_value'])
    .set_index(['location', 'as_of']))
   
    
df_hosp = (pd.read_parquet('surveillance data/observed data/truth_hosp.parquet')
           .assign(obs_value=lambda x: x['value'].where(x['value'] >= 0)).sort_values('target_end_date')
    .query("geo_type == 'state'").reset_index(drop=True))

df_hosp = df_hosp[(df_hosp.target_end_date>=pd.to_datetime('2021-01-01')) &\
                  (df_hosp.target_end_date<=pd.to_datetime('2022-12-31'))]

df_hosp['target_end_date'] = pd.to_datetime(df_hosp['target_end_date'])



df_hosp2 = (pd.read_parquet('surveillance data/observed data/COVID19-observed.parquet')
           .assign(obs_value=lambda x: x['obs_value'].where(x['value'] >= 0)).sort_values('target_end_date')
    .reset_index(drop=True))
df_hosp2['target_end_date'] = pd.to_datetime(df_hosp2['target_end_date'])

df_hosp2 = df_hosp2[(df_hosp2.target_end_date>=pd.to_datetime('2023-01-01')) &\
                  (df_hosp2.target_end_date<=pd.to_datetime('2024-05-04'))]

df_hosp = pd.concat([df_hosp, df_hosp2])


forecast_dates_hosp = df_hosp['target_end_date'][df_hosp['target_end_date'].dt.dayofweek == 0].unique()


baseline_model_all = pd.DataFrame()
for date in forecast_dates_hosp:
    print(date)
    dfsubmitted = pd.read_csv(f'https://raw.githubusercontent.com/reichlab/covid19-forecast-hub/refs/heads/master/data-processed/COVIDhub-baseline/{pd.to_datetime(date).strftime("%Y-%m-%d")}-COVIDhub-baseline.csv')
    dfsubmitted = dfsubmitted[dfsubmitted.target.isin(["1 day ahead inc hosp", "7 day ahead inc hosp", 
                                                       "14 day ahead inc hosp",  "21 day ahead inc hosp"])]
    dfsubmitted['quantile'] = dfsubmitted['quantile'].astype(str) 
    dfsubmitted['location'] = dfsubmitted['location'].astype(str)

    dfsubmitted = dfsubmitted.merge(locationsdf, on='location')

    for location in observations.location.unique()  :
        
        if location in ['GU', 'MP', 'AS', ]:
            continue

        forecast_date = pd.to_datetime(date)
        week_end = Week.fromdate(forecast_date).enddate() 

    
       
        #  filter surveillance data for chosen location and for all time points prior to forecast date
     
        
        try:
            obs_filt = observations_idx.loc[(location, forecast_date)]
        except KeyError:
            continue

        obs_filt = (obs_filt.loc[obs_filt['time_value'] <= forecast_date - timedelta(days=0)]
            .sort_values('time_value').dropna(subset=['value']).reset_index(drop=True))

        if len(obs_filt)==0:
            print('no surveillance data')


        #  get list of differences between successive weeks from historical incidence data
            # window_size = how much historical data to use. None = use all historical data. 
            # symmetrize: (logical) True = look at differences and their negatives to get symmetric distribution
            
        window_size = None
        symmetrize=True

        if window_size is None:
            window_size = len(obs_filt) - 1
        if window_size >= len(obs_filt):
            window_size = len(obs_filt) - 1

        diffs = np.diff(obs_filt['value'])[-window_size:]
        diffs = diffs[~np.isnan(diffs)]

        if symmetrize:
            inc_diffs = np.concatenate([diffs, -diffs])
        else:
            inc_diffs = list(diffs)

        inc_diffs = np.array(inc_diffs)


        # compute quantiles of differences distribution and ensure the median difference is 0. 

        num_samples = 100000

        sampled_inc_diffs = np.quantile(inc_diffs, np.linspace(0, 1, num_samples))

        median_diff = np.median(sampled_inc_diffs)
        sampled_inc_diffs -= median_diff


        # add differences to last observed surveillance data point and ensure non-negative incidence values

        inc_data = obs_filt['value']
        
        if use_submitted_median == True:
            if len(dfsubmitted[(dfsubmitted.abbreviation==location)])==0:
                continue
            last_observed_value = dfsubmitted[(dfsubmitted.abbreviation==location) &\
                                              (dfsubmitted['quantile']=='0.5')].value.unique()[0]
            last_observed_value = np.log(last_observed_value+1)
        else:
            last_observed_value = inc_data.iloc[-1]


        sampled_inc_raw = last_observed_value + sampled_inc_diffs
        sampled_inc_raw = np.maximum(sampled_inc_raw, 0)


        #calculate quantiles from sampled incidence values and save first horizon

        quantiles = [0.01, 0.025] + list(np.arange(0.05, 0.96, 0.05)) + [0.975, 0.99]

        predictions = np.quantile(sampled_inc_raw, quantiles, axis=0)

        results = pd.DataFrame({'horizon':1, 'forecast_date':pd.to_datetime(forecast_date),
                                'target_end_date': forecast_date + pd.Timedelta(days=0)   ,
                                'target':'inc hosp','location':location ,'quantile':quantiles, 'value':predictions})

       
        #  get predictions for larger horizons

        horizon = 7*12 # number of days ahead to predict

        rows = []
        rows.append(results)
        
        for h in list(range(2, horizon+1)):
            # Sample from differences again with replacement for each new horizon 
            # this step adds uncertainty at each horizon
            sampled_inc_diffs = np.random.choice(sampled_inc_diffs, size=num_samples, replace=True)
            sampled_inc_raw += sampled_inc_diffs # add to previous horizon incidence counts

            # Adjust incidence if symmetrize is TRUE. ensures median is same as last observed surveillance
            if symmetrize:
                sampled_inc_corrected = sampled_inc_raw - (np.median(sampled_inc_raw) - last_observed_value)
            else:
                sampled_inc_corrected = sampled_inc_raw

            # Ensure non-negative incidence
            sampled_inc_corrected = np.maximum(sampled_inc_corrected, 0)

            predictionshorizon = np.quantile(sampled_inc_corrected, quantiles, axis=0) # compute quantiles

            # save incidence for new horizon
            resultshorizon = pd.DataFrame({'horizon':h, 'forecast_date':pd.to_datetime(forecast_date),
                                            'target_end_date':forecast_date+ pd.Timedelta(days=h),
                                            'target':'inc hosp','location':location , 'quantile':quantiles, 
                                            'value':predictionshorizon})

            rows.append(resultshorizon)
        
            
        baseline_model = pd.concat(rows, ignore_index=True)

        baseline_model_all = pd.concat([baseline_model_all, baseline_model], ignore_index=True)
        
        
baseline_model_all['quantile'] = baseline_model_all['quantile'].round(3)  
baseline_model_all['value'] = baseline_model_all['value'].apply(lambda x: np.exp(x))       
        
baseline_model_all.to_parquet('benchmark forecasts/random walk baseline/covid_dailyhosp_log_baseline_h12.pq')

2021-01-04T00:00:00.000000000
2021-01-11T00:00:00.000000000
2021-01-18T00:00:00.000000000
2021-01-25T00:00:00.000000000
2021-02-01T00:00:00.000000000
2021-02-08T00:00:00.000000000
2021-02-15T00:00:00.000000000
2021-02-22T00:00:00.000000000
2021-03-01T00:00:00.000000000
2021-03-08T00:00:00.000000000
2021-03-15T00:00:00.000000000
2021-03-22T00:00:00.000000000
2021-03-29T00:00:00.000000000
2021-04-05T00:00:00.000000000
2021-04-12T00:00:00.000000000
2021-04-19T00:00:00.000000000
2021-04-26T00:00:00.000000000
2021-05-03T00:00:00.000000000
2021-05-10T00:00:00.000000000
2021-05-17T00:00:00.000000000
2021-05-24T00:00:00.000000000
2021-05-31T00:00:00.000000000
2021-06-07T00:00:00.000000000
2021-06-14T00:00:00.000000000
2021-06-21T00:00:00.000000000
2021-06-28T00:00:00.000000000
2021-07-05T00:00:00.000000000
2021-07-12T00:00:00.000000000
2021-07-19T00:00:00.000000000
2021-07-26T00:00:00.000000000
2021-08-02T00:00:00.000000000
2021-08-09T00:00:00.000000000
2021-08-16T00:00:00.000000000
2021-08-23

In [ ]:
covdailyhosp = pd.read_parquet('benchmark forecasts/random walk baseline/covid_dailyhosp_log_baseline_h12.pq').rename(columns={'location':'abbreviation'})

locationsdf = pd.read_csv('dat/locations.csv')
locationsdf['location'] = locationsdf['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 
locationsdf = locationsdf[['abbreviation', 'location', 'location_name']]

covdailyhosp = covdailyhosp.merge(locationsdf, on=['abbreviation'])

covdailyhosp['location'] = covdailyhosp['location'].astype(str)
covdailyhosp['location'] = covdailyhosp['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 

for col in ["location_name", "target", "abbreviation", ]:
    covdailyhosp[col] = covdailyhosp[col].astype("category")
    
covdailyhosp["value"] = covdailyhosp["value"].astype("float32")

covdailyhosp.to_parquet('benchmark forecasts/random walk baseline/covid_dailyhosp_log_baseline_h12.pq')

# COVID weekly hosp

In [4]:

# get historical data, current and past 

use_submitted_median = True

locationsdf = pd.read_csv('dat/locations.csv')
locationsdf['location'] = locationsdf['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 


df_hosp_versioned = pd.read_parquet('surveillance data/versioned_data/covid_hosp_24-25.parquet')

df_hosp_versioned_agg = (df_hosp_versioned.assign(value=lambda x: x['value'].where(x['value'] >= 0))
    .assign(target_end_date=lambda x: pd.to_datetime(x['target_end_date']).dt.to_period('W-SAT').dt.to_timestamp('W-SAT'))
    .groupby(['as_of', 'geo_value', 'target_end_date'], dropna=False).agg({'value': 'sum'}).reset_index()
    .sort_values('geo_value').reset_index(drop=True))

observations = df_hosp_versioned_agg.rename(columns={'geo_value':'location'})
observations1 = observations[(observations.target_end_date<=pd.to_datetime('2024-05-04')) ]
observations2 = observations[observations.target_end_date>=pd.to_datetime('2024-11-11')]
observations = pd.concat([observations1, observations2])

observations['week_end'] = observations['target_end_date'].apply(lambda x: Week.fromdate(x).enddate())
observations['location'] = observations['location'].apply(lambda x: str.upper(x))

observations['value'] = observations['value'].apply(lambda x: np.log(x+1))

      
forecast_dates_hosp = ['2024-11-23', '2024-11-30', '2024-12-07', '2024-12-14', '2024-12-21', '2024-12-28', 
                       '2025-01-04', '2025-01-11', '2025-01-18', '2025-02-01', '2025-02-08', '2025-02-15', 
                       '2025-02-22', '2025-03-01', '2025-03-08', '2025-03-15', '2025-03-22', '2025-03-29', 
                       '2025-04-05', '2025-04-12', '2025-04-19', '2025-04-26', '2025-05-03', '2025-05-10', 
                       '2025-05-17', '2025-05-24', '2025-05-31', '2025-06-07', '2025-06-14', '2025-06-21', 
                       '2025-06-28', '2025-07-05', '2025-07-12', '2025-07-19', '2025-07-26', '2025-08-02', 
                       '2025-08-09', '2025-08-16', '2025-08-23', '2025-08-30', '2025-09-06', '2025-09-13',
                       '2025-09-20', '2025-09-27', '2025-11-22', '2025-11-29', '2025-12-06', '2025-12-13',
                       '2025-12-20', '2025-12-27'] 

#  choose forecast date and location. 

baseline_model_all = pd.DataFrame()
for date in forecast_dates_hosp:
    print(date)
    for location in observations.location.unique()  :
        
        if location in ['GU', 'MP', 'VI', 'AS']:
            continue

        forecast_date = pd.to_datetime(date)
        week_end = Week.fromdate(forecast_date).enddate() 

        
        # filter surveillance data for chosen location and for all time points prior to forecast date

        obs_filt = observations[(observations['location'] == location) &\
                        (observations['target_end_date'] < forecast_date ) &\
                        (observations['as_of'] == forecast_date - timedelta(days=3) )].sort_values(by='week_end',ascending=True)
            

        if len(obs_filt)==0:
            print('no surveillance data')


        # Step 4: get list of differences between successive weeks from historical incidence data
            # window_size = how much historical data to use. None = use all historical data. 
            # symmetrize: (logical) True = look at differences and their negatives to get symmetric distribution
            
        window_size = None
        symmetrize=True

        if window_size is None:
            window_size = len(obs_filt) - 1
        if window_size >= len(obs_filt):
            window_size = len(obs_filt) - 1

        diffs = np.diff(obs_filt['value'])[-window_size:]
        diffs = diffs[~np.isnan(diffs)]

        if symmetrize:
            inc_diffs = list(diffs) + [-x for x in diffs]
        else:
            inc_diffs = list(diffs)

        inc_diffs = np.array(inc_diffs)


        # compute quantiles of differences distribution and ensure the median difference is 0. 

        num_samples = 100000

        sampled_inc_diffs = np.quantile(inc_diffs, np.linspace(0, 1, num_samples))

        median_diff = np.median(sampled_inc_diffs)
        sampled_inc_diffs -= median_diff


        # add differences to last observed surveillance data point and ensure non-negative incidence values

        inc_data = obs_filt['value']
        
        if use_submitted_median == True:
            df = pd.read_csv(f'https://raw.githubusercontent.com/CDCgov/covid19-forecast-hub/refs/heads/main/model-output/CovidHub-baseline/{forecast_date.strftime("%Y-%m-%d")}-CovidHub-baseline.csv')
            df = df.merge(locationsdf, on='location')
            df = df[df.horizon>=0]
            df['output_type_id'] = df['output_type_id'].astype(str)
            last_observed_value = df[(df.abbreviation==location) & (df['output_type_id']=='0.5')].value.unique()[0]
            last_observed_value = np.log(last_observed_value+1)
        else:
            last_observed_value = inc_data.iloc[-1]

       

        sampled_inc_raw = last_observed_value + sampled_inc_diffs
        sampled_inc_raw = np.maximum(sampled_inc_raw, 0)


        # calculate quantiles from sampled incidence values and save first horizon

        quantiles = [0.01, 0.025] + list(np.arange(0.05, 0.96, 0.05)) + [0.975, 0.99]

        baseline_model = pd.DataFrame()

        predictions = np.quantile(sampled_inc_raw, quantiles, axis=0)

        results = pd.DataFrame({'horizon':1, 'forecast_date':pd.to_datetime(forecast_date),
                                'target_end_date': week_end- pd.Timedelta(days=7) + timedelta(weeks=1)  ,
                                'target':'inc hosp','location':location ,'quantile':quantiles, 'value':predictions})

        baseline_model = pd.concat([baseline_model, results])


        # get predictions for larger horizons

        horizon = 12 # number of weeks ahead to predict

        for h in list(range(2, horizon+1)):
            # Sample from differences again with replacement for each new horizon 
            # this step adds uncertainty at each horizon
            sampled_inc_diffs = np.random.choice(sampled_inc_diffs, size=num_samples, replace=True)
            sampled_inc_raw = sampled_inc_raw + sampled_inc_diffs # add to previous horizon incidence counts

            # Adjust incidence if symmetrize is TRUE. ensures median is same as last observed surveillance
            if symmetrize:
                sampled_inc_corrected = sampled_inc_raw - (np.median(sampled_inc_raw) - last_observed_value)
            else:
                sampled_inc_corrected = sampled_inc_raw

            # Ensure non-negative incidence
            sampled_inc_corrected = np.maximum(sampled_inc_corrected, 0)

            predictionshorizon = np.quantile(sampled_inc_corrected, quantiles, axis=0) # compute quantiles

            # save incidence for new horizon
            resultshorizon = pd.DataFrame({'horizon':h, 'forecast_date':pd.to_datetime(forecast_date),
                                            'target_end_date':week_end- pd.Timedelta(days=7) + timedelta(weeks=1*h),
                                            'target':'inc hosp','location':location , 'quantile':quantiles, 
                                            'value':predictionshorizon})

            baseline_model = pd.concat([baseline_model, resultshorizon])

        baseline_model['quantile'] = baseline_model['quantile'].round(3)  

        baseline_model_all = pd.concat([baseline_model_all, baseline_model])
        
baseline_model_all['value'] = baseline_model_all['value'].apply(lambda x: np.exp(x))       
        
baseline_model_all.to_parquet('benchmark forecasts/random walk baseline/covid_hosp_log_baseline_2025_h12.pq')

2024-11-23
2024-11-30
2024-12-07
2024-12-14
2024-12-21
2024-12-28
2025-01-04
2025-01-11
2025-01-18
2025-02-01
2025-02-08
2025-02-15
2025-02-22
2025-03-01
2025-03-08
2025-03-15
2025-03-22
2025-03-29
2025-04-05
2025-04-12
2025-04-19
2025-04-26
2025-05-03
2025-05-10
2025-05-17
2025-05-24
2025-05-31
2025-06-07
2025-06-14
2025-06-21
2025-06-28
2025-07-05
2025-07-12
2025-07-19
2025-07-26
2025-08-02
2025-08-09
2025-08-16
2025-08-23
2025-08-30
2025-09-06
2025-09-13
2025-09-20
2025-09-27
2025-11-22
2025-11-29
2025-12-06
2025-12-13
2025-12-20
2025-12-27


In [ ]:
covhosp = pd.read_parquet('benchmark forecasts/random walk baseline/covid_hosp_log_baseline_2025_h12.pq').rename(columns={'location':'abbreviation'})

locationsdf = pd.read_csv('dat/locations.csv')
locationsdf['location'] = locationsdf['location'].apply(lambda x: '0'+x if x!='US' and int(x)<10 else x) 
locationsdf = locationsdf[['abbreviation', 'location', 'location_name']]

covhosp = covhosp.merge(locationsdf, on=['abbreviation']).rename(columns={'forecast_date':'reference_date'})

covhosp.to_parquet('benchmark forecasts/random walk baseline/covid_hosp_log_baseline_2025_h12.pq')

# ILI 

In [19]:
import pickle
from scipy.special import expit, logit

def get_log_binned_baseline(dates, locations):
   # Step 1: get historical data, current and past  

    observations = pd.read_parquet('surveillance data/observed data/ilinet_truth.parquet')[['region', 'year', 'week', 'weighted_ili',
                                                                           'week_start', 'location']]

       
    ili_versioned = pd.read_parquet("surveillance data/versioned_data/influenza_ili_new.parquet")
    ili_versioned['as_of'] = pd.to_datetime(ili_versioned['as_of'])
    ili_versioned['as_of_start_date'] = ili_versioned['as_of'].apply(lambda row: Week.fromdate(row).startdate())
    ili_versioned['as_of_end_date'] = ili_versioned['as_of'].apply(lambda row: Week.fromdate(row).enddate())

    ili_versioned = ili_versioned.rename(columns={'wili': 'weighted_ili','as_of_start_date': 'week_start',
                                                 'as_of_end_date': 'week_end','geo_value':'region'})


    ili_name_map = pd.DataFrame({'region': ili_versioned['region'].unique()[[10, 0, 2, 3, 4, 5, 6, 7, 8, 
                                                              9, 1]], 'location': observations['location'].unique()})

    ili_versioned = ili_versioned.merge(ili_name_map, on='region')

    ili_versioned['weighted_ili'] = ili_versioned['weighted_ili'].apply(lambda x: logit((x/100)+0.001) if x==0 else logit(x/100) )

   
    
    # get bins
   
    with open('dat/ILI_bin_edges_dict.pkl', 'rb') as pickle_file: 
        bin_edges_dict = pickle.load(pickle_file)

   # create baseline model
    
    
    num_samples = 100000
    horizon_max = 12
    window_size_param = None
    symmetrize = True

    # Collect rows in a list, concat ONCE at the end
    all_results = []

    # Pre-compute bin edges per season (they don't depend on date/location)
    season_bin_edges = {
        s: np.array([bin_edge_season[0][0]] + [b[1] for b in bin_edge_season])
        for s, bin_edge_season in bin_edges_dict.items()
    }
    season_bin_starts = {s: [b[0] for b in bin_edges_dict[s]] for s in bin_edges_dict}
    season_bin_ends   = {s: [b[1] for b in bin_edges_dict[s]] for s in bin_edges_dict}

    # Pre-index surveillance data for fast lookup
    ili_indexed = ili_versioned.set_index(['location', 'as_of']).sort_index()


    for date in dates:
        print(date)
        forecast_date = pd.to_datetime(date)

        season = (f'{forecast_date.year}-{forecast_date.year+1}'
                  if (forecast_date.week >= 30) or (forecast_date.month>=11)
                  else f'{forecast_date.year-1}-{forecast_date.year}')

        bin_edges    = season_bin_edges[season]
        bin_starts   = season_bin_starts[season]
        bin_ends     = season_bin_ends[season]
        n_bins       = len(bin_starts)

        for location in locations:
            try:
                obs_filt = ili_indexed.loc[(location, date)].sort_values('target_end_date')
            except KeyError:
                print('no surveillance data')
                continue

            if isinstance(obs_filt, pd.Series):
                obs_filt = obs_filt.to_frame().T

            ili_values = obs_filt['weighted_ili'].to_numpy()
            if len(ili_values) < 2:
                continue

            # Build difference distribution
            window_size = window_size_param if window_size_param is not None else len(ili_values) - 1
            window_size = min(window_size, len(ili_values) - 1)

            diffs = np.diff(ili_values)[-window_size:]
            diffs = diffs[~np.isnan(diffs)]
            if symmetrize:
                diffs = np.concatenate([diffs, -diffs])
            

            # compute quantiles of differences distribution and ensure the median difference is 0. 


            sampled_inc_diffs = np.quantile(diffs, np.linspace(0, 1, num_samples))

            median_diff = np.median(sampled_inc_diffs)
            sampled_inc_diffs -= median_diff



            #  add differences to last observed surveillance data point and ensure non-negative incidence values

            last_observed_value = ili_values[-1]
            sampled_inc_raw = last_observed_value + sampled_inc_diffs

            
            proportions = np.clip(expit(sampled_inc_raw), 0, 1)
            counts, _ = np.histogram(proportions, bins=bin_edges)
            total = counts.sum()
            probs = counts / total if total > 0 else counts

            all_results.append(pd.DataFrame({
                'horizon': 1, 'forecast_date': forecast_date, 'season': season,
                'target': 'ILI', 'location': location,
                'bin_start_incl': bin_starts, 'bin_end_notincl': bin_ends,
                'value': probs,
            }))
            
            
            #  get predictions for larger horizons

            for h in list(range(2, horizon_max+1)):
                # Sample from differences again with replacement for each new horizon 
                # this step adds uncertainty at each horizon
                sampled_inc_diffs = np.random.choice(sampled_inc_diffs, size=num_samples, replace=True)
                sampled_inc_raw = sampled_inc_raw + sampled_inc_diffs # add to previous horizon incidence counts

                # Adjust incidence if symmetrize is TRUE. ensures median is same as last observed surveillance
                if symmetrize:
                    sampled_inc_corrected = sampled_inc_raw - (np.median(sampled_inc_raw) - last_observed_value)
                else:
                    sampled_inc_corrected = sampled_inc_raw

                proportions = np.clip(expit(sampled_inc_corrected), 0, 1)
                counts, _ = np.histogram(proportions, bins=bin_edges)
                total = counts.sum()
                probs = counts / total if total > 0 else counts

                all_results.append(pd.DataFrame({
                    'horizon': h, 'forecast_date': forecast_date, 'season': season,
                    'target': 'ILI', 'location': location,
                    'bin_start_incl': bin_starts, 'bin_end_notincl': bin_ends,
                    'value': probs,
                }))

    baseline_model_all = pd.concat(all_results, ignore_index=True)



    baseline_model_all['horizon'] = baseline_model_all['horizon'] - 1
    baseline_model_all['target_end_date'] =  baseline_model_all['forecast_date'].apply(lambda key: Week.fromdate(key).startdate()  ) 
    baseline_model_all['target_end_date'] =  baseline_model_all['target_end_date'] + baseline_model_all['horizon' ].apply(lambda x: pd.Timedelta(days = x*7 - 1))
    baseline_model_all['target_end_date'] =  baseline_model_all['target_end_date'].apply(lambda key: Week.fromdate(key).enddate())

    baseline_model_all['horizon'] = baseline_model_all['horizon'] - 1
    return baseline_model_all

In [21]:
locations = ['US National', 'HHS Region 1', 'HHS Region 2', 'HHS Region 3','HHS Region 4', 'HHS Region 5', 
             'HHS Region 6', 'HHS Region 7', 'HHS Region 8', 'HHS Region 9', 'HHS Region 10']


forecasts_all = pd.read_csv('dat/ILIforecasts_all.csv')
dates = list(pd.read_csv('dat/ILI_ensemble_forecast_dates.csv').ens_forecast_date.unique())

#baseline_quantiles = get_quantile_baseline(dates, locations)
baseline_binned = get_log_binned_baseline(dates, locations)
baseline_binned.to_parquet('benchmark forecasts/random walk baseline/ILI_log_baseline_h12_new.pq')

2015-11-02
2015-11-09
2015-11-16
2015-11-23
2015-12-01
2015-12-07
2015-12-14
2015-12-21
2015-12-30
2016-01-06
2016-01-11
2016-01-18
2016-01-25
2016-02-01
2016-02-08
2016-02-15
2016-02-22
2016-02-29
2016-03-07
2016-03-14
2016-03-21
2016-03-28
2016-04-04
2016-04-11
2016-04-18
2016-04-25
2016-05-02
2016-05-09
2016-05-16
2016-11-07
2016-11-14
2016-11-21
2016-11-28
2016-12-05
2016-12-12
2016-12-19
2016-12-23
2017-01-02
2017-01-09
2017-01-17
2017-01-23
2017-01-30
2017-02-06
2017-02-13
2017-02-21
2017-02-27
2017-03-06
2017-03-14
2017-03-21
2017-03-28
2017-04-04
2017-04-11
2017-04-18
2017-04-24
2017-05-01
2017-05-09
2017-05-15
2017-11-06
2017-11-13
2017-11-20
2017-11-29
2017-12-04
2017-12-11
2017-12-18
2017-12-27
2018-01-03
2018-01-08
2018-01-16
2018-01-22
2018-01-29
2018-02-05
2018-02-12
2018-02-20
2018-02-26
2018-03-05
2018-03-13
2018-03-20
2018-03-26
2018-04-03
2018-04-10
2018-04-17
2018-04-24
2018-05-01
2018-05-08
2018-05-14
2018-10-29
2018-11-04
2018-11-11
2018-11-20
2018-11-27
2018-12-04